# Simple RAG with Supabase pgvector

In this notebook, you'll build a RAG pipeline using **Supabase** as your vector database and **OpenAI** for embeddings and chat. By the end, you'll be able to store documents, search them by semantic similarity, and have an LLM answer questions grounded in your own data.

In [ ]:
%pip install supabase pgvector vecs openai python-dotenv --upgrade

## 1. Setup

First, install the required packages and load your environment variables. Make sure your `.env` file contains `SUPABASE_URL` and `SUPABASE_KEY`. If `OPENAI_API_KEY` is not already set in your environment, the notebook will prompt for it securely.


In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

True

In [1]:
from supabase import create_client, Client
import os

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")
supabase: Client = create_client(url, key)

## 2. Generate Embeddings

Next, you'll create a helper function that turns any text into a 1536-dimensional vector using OpenAI's `text-embedding-3-small` model. These vectors are what allow you to compare documents by meaning rather than exact keywords.

In [ ]:
import getpass
import os
from openai import OpenAI

# API Key Setup
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

# Initialize OpenAI client
client = OpenAI()
MODEL = "gpt-5-mini"

# Function to get the vector embedding for a given text
def get_vector_embeddings(text):
    response = client.embeddings.create(
        input=text,
        model="text-embedding-3-small"  # Using latest embedding model
    )
    embeddings = response.data[0].embedding
    return embeddings


query_embedding = get_vector_embeddings("Your text string goes here")
print(len(query_embedding))
query_embedding


## 3. Create the Items Table in Supabase

Before you can store embeddings, you need a table to hold them. Run the following SQL in your **Supabase SQL Editor** to create the `items` table:

```sql
CREATE TABLE items (
    id         SERIAL PRIMARY KEY,
    embedding  vector(1536),
    text       TEXT
);
```

Each row will store a chunk of text alongside its vector embedding.

## 4. Chunk and Store Your Documents

Now you'll take a sample document (a fictional employee handbook), split it into paragraphs, generate an embedding for each one, and insert them into your Supabase `items` table.

In [ ]:
employee_handbook = """
Welcome to the "Unicorn Enterprises: Where Magic Happens" Employee Handbook! We're thrilled to have you join our team of dreamers, doers, and unicorn enthusiasts. At Unicorn Enterprises, we believe that work should be as enchanting as it is productive. This handbook is your ticket to the magical world of our company, where we'll outline the principles, policies, and practices that guide us on this extraordinary journey. So, fasten your seatbelts and get ready to embark on an adventure like no other!

**1: Our Magical Culture**
At Unicorn Enterprises, we take pride in our unique and enchanting company culture. We believe that creativity and innovation flourish best when people are happy and inspired. From our weekly "Wear Your Favorite Mythical Creature Costume" day on Fridays to our in-house unicorn petting zoo, we aim to infuse magic into every corner of our workplace. So, don't be surprised if you find a fairy tale book in the breakroom or a gnome guiding you to the restroom. Our culture is designed to spark your imagination and encourage collaboration among our magical team.

**2: Unicorn Code of Conduct**
While we embrace creativity, we also value professionalism. Our Unicorn Code of Conduct ensures that we maintain a harmonious and respectful environment. Treating all team members, regardless of their unicorn species, with kindness and respect is essential. We also encourage open communication and constructive feedback because, in our world, every opinion matters, just like every horn on a unicorn's head!

**3: Magical Work-Life Balance**
At Unicorn Enterprises, we understand the importance of maintaining a balanced life. We offer flexible work hours, magical mental health days, and even an on-site wizard to provide stress-relief spells when needed. We believe that a happy and well-rested employee is a creative and productive employee. So, don't hesitate to use our relaxation chambers or join a group meditation session under the office rainbow.

**4: Enchanted Benefits**
Our commitment to your well-being extends to our magical benefits package. You'll enjoy a treasure chest of perks, including unlimited unicorn rides, a bottomless cauldron of coffee and potions, and access to our company library filled with spellbinding books. We also offer competitive health and dental plans, ensuring your physical well-being is as robust as your magical spirit.

**5: Continuous Learning and Growth**
At Unicorn Enterprises, we believe in continuous learning and growth. We provide access to a plethora of online courses, enchanted workshops, and wizard-led training sessions. Whether you're aspiring to master new spells or conquer new challenges, we're here to support your personal and professional development.
As we conclude this handbook, remember that at Unicorn Enterprises, the pursuit of excellence is a never-ending quest. Our company's success depends on your passion, creativity, and commitment to making the impossible possible. We encourage you to always embrace the magic within and outside of work, and to share your ideas and innovations to keep our enchanted journey going. Thank you for being a part of our mystical family, and together, we'll continue to create a world where magic and business thrive hand in hand!
"""

# Split the employee handbook into paragraphs
chunks = [p.strip() for p in employee_handbook.split('\n\n') if p.strip()]
print(f"Number of paragraphs: {len(chunks)}")

# Print each paragraph with its index
for i, p in enumerate(chunks):
    print(f"\nParagraph {i}:")
    print(p)

In [10]:
# Get embeddings for each chunk
embeddings = []
for chunk in chunks:
    embedding = get_vector_embeddings(chunk)
    embeddings.append(embedding)

# Insert chunks and their embeddings into Supabase
for chunk, embedding in zip(chunks, embeddings):
    data = supabase.table('items').insert({
        "text": chunk,
        "embedding": embedding
    }).execute()
    
print("Successfully inserted all chunks into Supabase")


Successfully inserted all chunks into Supabase


## 5. Create a Similarity Search Function in Supabase

To query your vectors, you'll create a PostgreSQL function that uses pgvector's **cosine distance** operator (`<=>`). Run the following SQL in your **Supabase SQL Editor**:

```sql
-- Ensure the pgvector extension is enabled
CREATE EXTENSION IF NOT EXISTS vector;

-- Function: find the most similar items to a query embedding
CREATE OR REPLACE FUNCTION match_items (
    query_embedding  vector(1536),  -- Must match your embedding dimensions
    match_threshold  float,         -- Minimum similarity score (e.g. 0.4)
    match_count      int            -- Max number of results to return
)
RETURNS TABLE (
    id          bigint,
    text        text,
    similarity  float
)
LANGUAGE sql STABLE
AS $$
    SELECT
        items.id,
        items.text,
        1 - (items.embedding <=> query_embedding) AS similarity
    FROM items
    WHERE 1 - (items.embedding <=> query_embedding) > match_threshold
    ORDER BY items.embedding <=> query_embedding
    LIMIT match_count;
$$;
```

**How it works:**
- The `<=>` operator computes the **cosine distance** between two vectors (lower = more similar).
- You convert distance to a similarity score with `1 - distance`, so values closer to **1.0** mean a stronger match.
- `match_threshold` filters out weak matches, and `match_count` caps your results.

## 6. Test the Similarity Search

With your function in place, you can now call it from Python using Supabase's `.rpc()` method. Try asking a question and see which document chunks come back as the closest matches.

In [42]:
# Your query text
query_text = "Do we get free Unicorn rides?"

# Generate the embedding for the query text
query_embedding = get_vector_embeddings(query_text)

# Call the database function using rpc
try:
    results = supabase.rpc('match_items', {
        'query_embedding': query_embedding,
        'match_threshold': 0.4,  # Adjust this threshold as needed
        'match_count': 2         # Number of results you want
    }).execute()

    # Process the results
    # Print each result with its similarity score
    for result in results.data:
        print(f"\nDocument ID: {result['id']}")
        print(f"Similarity Score: {result['similarity']:.3f}")
        print("Text:")
        print(result['text'])
        print("-" * 80)

except Exception as e:
    print(f"An error occurred: {e}")
    # Check if the error message gives clues, e.g., function not found



Document ID: 2
Similarity Score: 0.503
Text:
**1: Our Magical Culture**
At Unicorn Enterprises, we take pride in our unique and enchanting company culture. We believe that creativity and innovation flourish best when people are happy and inspired. From our weekly "Wear Your Favorite Mythical Creature Costume" day on Fridays to our in-house unicorn petting zoo, we aim to infuse magic into every corner of our workplace. So, don't be surprised if you find a fairy tale book in the breakroom or a gnome guiding you to the restroom. Our culture is designed to spark your imagination and encourage collaboration among our magical team.
--------------------------------------------------------------------------------

Document ID: 5
Similarity Score: 0.494
Text:
**4: Enchanted Benefits**
Our commitment to your well-being extends to our magical benefits package. You'll enjoy a treasure chest of perks, including unlimited unicorn rides, a bottomless cauldron of coffee and potions, and access to our

## 7. Build a Reusable Retrieve Function

Let's wrap the search logic into a clean `retrieve()` function you can reuse throughout your project.

In [43]:
def retrieve(query_text, match_count=2):
    # Get vector embedding for the query text
    query_embedding = get_vector_embeddings(query_text)
    
    # Query using pgvector's cosine distance search
    response = supabase.rpc(
        'match_items',  # Use vector similarity search
        {
            'query_embedding': query_embedding,
            'match_threshold': 0.4,  # Similarity threshold
            'match_count': match_count  # Number of matches to return
        }
    ).execute()
    
    # Process and format results
    results = []
    for result in response.data:
        results.append({
            'id': result['id'],
            'similarity': result['similarity'],
            'text': result['text']
        })
    
    return results


query = "Do we get free Unicorn rides?"
retrieve(query)

[{'id': 2,
  'similarity': 0.502970903226465,
  'text': '**1: Our Magical Culture**\nAt Unicorn Enterprises, we take pride in our unique and enchanting company culture. We believe that creativity and innovation flourish best when people are happy and inspired. From our weekly "Wear Your Favorite Mythical Creature Costume" day on Fridays to our in-house unicorn petting zoo, we aim to infuse magic into every corner of our workplace. So, don\'t be surprised if you find a fairy tale book in the breakroom or a gnome guiding you to the restroom. Our culture is designed to spark your imagination and encourage collaboration among our magical team.'},
 {'id': 5,
  'similarity': 0.493960173017949,
  'text': "**4: Enchanted Benefits**\nOur commitment to your well-being extends to our magical benefits package. You'll enjoy a treasure chest of perks, including unlimited unicorn rides, a bottomless cauldron of coffee and potions, and access to our company library filled with spellbinding books. We a

## 8. Putting It All Together: RAG

The `search_and_chat()` function below:

1. **Retrieves** the most relevant chunks for your question using vector search.
2. **Injects** those chunks as context into a prompt.
3. **Sends** the prompt to an LLM, which answers based only on the retrieved context.

This is the core RAG pattern: retrieve first, then generate.

In [ ]:
def search_and_chat(chat_prompt, k=2):
    # Print the user query
    print("\nUser Query:")
    print(chat_prompt)
    print("-" * 80)
    
    # Perform the vector search using retrieve()
    search_results = retrieve(chat_prompt, k)
    
    # Print the retrieved context
    print("\nRetrieved Context:")
    for result in search_results:
        print(f"Document {result['id']} (Similarity: {result['similarity']:.3f})")
        print(result['text'])
        print()
    print("-" * 80)
    
    prompt_with_context = f"""Context: {search_results}\n\n
    Answer the question: {chat_prompt}"""

    # Create a list of messages for the chat
    messages = [
        {"role": "system", "content": "Please answer the questions provided by the user. Use only the context provided to you, if you don't know the answer say \"I don't know\"."},
        {"role": "user", "content": prompt_with_context},
    ]

    # Get the model's response
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    # Print the model's response
    print("\nAI Response:")
    print(response.choices[0].message.content)
    print("-" * 80)

# Example usage
search_and_chat("Do we get free Unicorn rides?")

## Summary

In this notebook you built a complete RAG pipeline:

- **Embeddings**: you used OpenAI's `text-embedding-3-small` model to convert text chunks into 1536-dimensional vectors.
- **Vector storage**: you stored those vectors in a Supabase Postgres table using the `pgvector` extension.
- **Similarity search**: you created a SQL function that finds the most relevant chunks using cosine distance.
- **RAG**: you combined retrieval with an LLM call so the model answers questions grounded in your own documents.

This same pattern works with much larger datasets. Swap in your own documents, tune the similarity threshold, and you have a working knowledge base.